# 01 – Effizienz: Werde ich bei gleichem Puls schneller?

**Leitfrage 1.** Wenn die aerobe Fitness steigt, läuft man bei gleicher Herzfrequenz schneller. Zwei Kennzahlen machen das messbar:

- **Effizienzfaktor (EF)** = mittlere Geschwindigkeit [m/min] ÷ mittlere Herzfrequenz. Höher ist besser.
- **Aerobe Entkopplung** = um wie viel Prozent der EF in der zweiten Hälfte eines langen, gleichmäßigen Laufs schlechter ist als in der ersten. Niedrig heißt: Die Herzfrequenz driftet bei gleichem Tempo kaum nach oben, die Ausdauerbasis trägt.

**Definitionen** (festgelegt am 27.10.2026, Code in `metrics.py`):

- Gerechnet wird nur über die **bewegte Zeit**, also Messpunkte ab 1,0 m/s. Ampelstopps zählen nicht.
- Die ersten **10 Minuten bewegte Zeit** sind Aufwärmen und fallen heraus.
- EF nur für Läufe ab **30 Minuten** bewegter Zeit, Entkopplung nur ab **45 Minuten** und bei gleichmäßigem Tempo (Variationskoeffizient der Geschwindigkeit unter 0,15).

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from myfitness import plots
from myfitness.config import PROCESSED

activities = pl.read_parquet(PROCESSED / "activities.parquet")
activity_metrics = pl.read_parquet(PROCESSED / "activity_metrics.parquet")
trend = plots.ef_trend_table(activity_metrics, activities)

print(f"{trend.height} Läufe mit EF, {activity_metrics['decoupling_pct'].is_not_null().sum()} mit Entkopplung")

## EF über das Jahr

Jeder Punkt ist ein Lauf, die Farbe zeigt die Temperatur beim Start. Die Linie ist das gleitende Mittel der letzten 28 Tage. Sie ist unterbrochen, wo mehr als 28 Tage kein Lauf ab 30 Minuten vorliegt.

In [ ]:
plots.plot_ef_over_time(trend)
plt.show()

Wie hat sich der EF zwischen Anfang und Ende des Zeitraums verändert? Verglichen werden die ersten und die letzten drei Monate.

In [ ]:
first_day = trend["start_local"].min()
last_day = trend["start_local"].max()
three_months = pl.duration(days=91)

periods = trend.with_columns(
    period=pl.when(pl.col("start_local") < pl.lit(first_day) + three_months)
    .then(pl.lit("erste 3 Monate"))
    .when(pl.col("start_local") > pl.lit(last_day) - three_months)
    .then(pl.lit("letzte 3 Monate"))
).filter(pl.col("period").is_not_null())

periods.group_by("period").agg(
    runs=pl.len(),
    ef_mean=pl.col("ef").mean(),
    ef_median=pl.col("ef").median(),
).sort("period")

## Störgröße Temperatur

Hitze belastet den Kreislauf: Bei gleichem Tempo steigt die Herzfrequenz, der EF sinkt. Der einfache Zusammenhang zeigt aber das Gegenteil:

In [ ]:
plots.plot_ef_vs_temperature(trend)
plt.show()

trend.select(r_ef_temperature=pl.corr("ef", "temp_c"))

Der Grund ist eine **Störgröße**: Warme Läufe fanden im Sommer statt, als die Form ohnehin besser war. Temperatur und Fitness-Trend sind miteinander vermischt.

Um sie zu trennen, schauen wir nicht auf den EF selbst, sondern auf `ef_vs_trend`, also die Abweichung jedes Laufs vom 28-Tage-Mittel seiner Zeit. Das ist eine einfache Form der Bereinigung um den Trend.

In [ ]:
trend.select(r_ef_vs_trend_temperature=pl.corr("ef_vs_trend", "temp_c"))

In [ ]:
heat_threshold_c = 22

trend.group_by(hot=pl.col("temp_c") >= heat_threshold_c).agg(
    runs=pl.len(),
    mean_ef_vs_trend=pl.col("ef_vs_trend").mean(),
).sort("hot")

## Aerobe Entkopplung

Nur lange, gleichmäßige Läufe. Unter 5 % gilt als Zeichen einer gut entwickelten Ausdauerbasis. Das ist eine Faustregel aus der Trainingslehre, keine feste Grenze. Negative Werte bedeuten, dass die zweite Hälfte effizienter war, etwa weil das Tempo am Ende anzog oder es kühler wurde.

In [ ]:
plots.plot_decoupling_over_time(activity_metrics, activities)
plt.show()

In [ ]:
# All long runs, including those excluded as uneven (steady = false)
(
    activity_metrics.filter(pl.col("speed_cv").is_not_null())
    .join(activities.select("activity_id", "start_local", "name", "distance_m", "temp_c"), on="activity_id")
    .select("start_local", "name", "distance_m", "temp_c", "speed_cv", "steady", "ef_first_half", "ef_second_half", "decoupling_pct")
    .sort("start_local")
)

## Grenzen der Aussage

- **n = 1, Beobachtungsdaten.** Der Anstieg des EF zeigt, *dass* sich etwas verändert hat, nicht *warum*: Training, Gewicht, Schuhe, Strecken oder Messungenauigkeit kommen alle in Frage.
- **Streckenprofil:** Höhenmeter sind nicht bereinigt. Hügelige Läufe drücken den EF.
- **Trainingsart:** Intervall- und Schwellenläufe ab 30 Minuten zählen mit. Ihr EF ist nicht direkt mit dem lockerer Dauerläufe vergleichbar.
- **Temperatur:** Die Abweichung vom Trend ist eine grobe Bereinigung. Mit nur wenigen heißen Läufen ist der Hitzeeffekt eine **Hypothese**, kein Beleg.
- **Saisonale Lücken:** Von Januar bis Juni gibt es wenige lange Läufe. Der Trend ist dort unsicher.
- **Entkopplung:** Die Basis sind wenige Läufe. Die Schwelle für „gleichmäßig“ (Variationskoeffizient 0.15) ist ein Startwert und lässt sich in `metrics.py` anpassen.